# Day 14 Tutorial：传统机器学习阶段报告

## Goal

审计 Day 13 已实际执行的教程逐折 CSV，验证 schema 和折完整性，生成可追溯汇总与一份明确标注为人工数据教程的报告预览。


## Setup

课程源只审计 Day 13 的课程 fixture，并写入本日 `tutorial_outputs/`；`learning_outputs/day14_ml_stage_report/` 中的个人副本只审计本人 Day 13 `results/fold_metrics.csv`，并写入自己的 `results/`。源文件缺失时会停止并给出下一条可执行命令，绝不补造分数。


In [1]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "requirements-learning.txt").is_file():
            return candidate
    raise FileNotFoundError("Could not locate repository root")

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(CURRENT_DIR)
COURSE_DIR = REPO_ROOT / "curriculum/core/day14_ml_stage_report"
LEARNER_DIR = REPO_ROOT / "learning_outputs/day14_ml_stage_report"

def repo_relative(path):
    return path.resolve().relative_to(REPO_ROOT).as_posix()

if CURRENT_DIR == COURSE_DIR:
    RUN_MODE = "course_tutorial"
    SOURCE_PATH = REPO_ROOT / "curriculum/core/day13_fair_comparison/tutorial_outputs/fold_metrics.csv"
    OUTPUT_DIR = COURSE_DIR / "tutorial_outputs"
    REPORT_LABEL = "人工教程"
    EXECUTION_OBSERVATION = "课程教程的同折比较、逐折保存、汇总与审计流程已经实际运行。"
    MISSING_EVIDENCE_NOTE = "- 尚未在本人实验目录运行并记录；"
    MISSING_SOURCE_HINT = (
        "请先运行：cd curriculum/core/day13_fair_comparison && "
        "python -m jupyter nbconvert --to notebook --execute --inplace tutorial.ipynb"
    )
elif CURRENT_DIR == LEARNER_DIR:
    RUN_MODE = "learner_workspace"
    SOURCE_PATH = REPO_ROOT / "learning_outputs/day13_fair_comparison/results/fold_metrics.csv"
    OUTPUT_DIR = LEARNER_DIR / "results"
    REPORT_LABEL = "个人运行的人工教程"
    EXECUTION_OBSERVATION = "本人已在个人目录运行人工教程，并完成同折结果的保存、汇总与审计。"
    MISSING_EVIDENCE_NOTE = "- 当前证据仍来自固定人工教学数据，不是研究数据；"
    MISSING_SOURCE_HINT = (
        "请先在仓库根目录运行 直接运行 Day 13 教程，并把个人记录写入 learning_outputs/day13_fair_comparison/，完成并执行个人 Day 13，"
        "确认 learning_outputs/day13_fair_comparison/results/fold_metrics.csv 已生成。"
    )
else:
    raise RuntimeError(
        "无法安全判断 Day 14 输入/输出目录。请在以下二选一目录中启动或执行 Notebook：\n"
        "- curriculum/core/day14_ml_stage_report\n"
        "- learning_outputs/day14_ml_stage_report\n"
        "若个人目录还不存在，先在仓库根目录运行：直接运行 Day 14 教程，并把个人记录写入 learning_outputs/day14_ml_stage_report/"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_PATH.is_file():
    raise FileNotFoundError(
        f"未找到 Day 13 逐折文件：{repo_relative(SOURCE_PATH)}\n{MISSING_SOURCE_HINT}"
    )

print({
    "python": platform.python_version(),
    "mode": RUN_MODE,
    "source": repo_relative(SOURCE_PATH),
    "output": repo_relative(OUTPUT_DIR),
})


{'python': '3.10.20', 'mode': 'course_tutorial', 'source': 'curriculum/core/day13_fair_comparison/tutorial_outputs/fold_metrics.csv', 'output': 'curriculum/core/day14_ml_stage_report/tutorial_outputs'}


## Steps

### 1. 读取源文件并审计结构


In [2]:
fold_metrics = pd.read_csv(SOURCE_PATH)
EXPECTED_MODELS = {"dummy", "ridge", "decision_tree", "random_forest", "gradient_boosting"}
required_columns = {"model", "fold", "split", "mae", "rmse", "r2"}
missing_columns = required_columns - set(fold_metrics.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if fold_metrics.duplicated(["model", "fold"]).any():
    raise ValueError("Duplicate model/fold rows found")
if set(fold_metrics["split"]) != {"cv_valid"}:
    raise ValueError("Unexpected split labels")
if len(fold_metrics) != 25:
    raise ValueError(f"Expected exactly 25 model/fold rows; found {len(fold_metrics)}")
if set(fold_metrics["model"]) != EXPECTED_MODELS:
    raise ValueError(
        f"Expected models {sorted(EXPECTED_MODELS)}; found {sorted(set(fold_metrics['model']))}"
    )
if not np.isfinite(fold_metrics[["mae", "rmse", "r2"]]).all().all():
    raise ValueError("Non-finite regression metric found")

expected_folds = {1, 2, 3, 4, 5}
fold_sets = fold_metrics.groupby("model")["fold"].apply(set)
if not fold_sets.apply(lambda value: value == expected_folds).all():
    raise ValueError("At least one model lacks the complete 1-5 fold set")

display(fold_metrics.head().round(4))


,model,fold,split,mae,rmse,r2
0,dummy,1,cv_valid,99.9577,123.3705,-0.2201
1,dummy,2,cv_valid,92.0413,115.3721,-0.0001
2,dummy,3,cv_valid,131.8765,155.9449,-0.0312
3,dummy,4,cv_valid,122.0917,158.1096,-0.0765
4,dummy,5,cv_valid,125.0139,167.4333,-0.0093


### 2. 从审计通过的逐折证据生成汇总


In [3]:
summary = (
    fold_metrics.groupby("model")
    .agg(
        mae_mean=("mae", "mean"),
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        r2_mean=("r2", "mean"),
        n_folds=("fold", "count"),
    )
    .reset_index()
    .sort_values("rmse_mean")
    .reset_index(drop=True)
)
summary["source_file"] = repo_relative(SOURCE_PATH)
summary["protocol"] = "deterministic_synthetic_random_kfold5_tutorial"
summary_path = OUTPUT_DIR / "ml_stage_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary.round(4))


,model,mae_mean,rmse_mean,rmse_std,r2_mean,n_folds,source_file,protocol
0,ridge,14.9268,18.6088,2.7173,0.9796,5,curriculum/core/day13_fair_comparison/tutorial...,deterministic_synthetic_random_kfold5_tutorial
1,gradient_boosting,58.2607,75.1281,13.2315,0.7085,5,curriculum/core/day13_fair_comparison/tutorial...,deterministic_synthetic_random_kfold5_tutorial
2,random_forest,68.5626,91.5439,14.4172,0.5454,5,curriculum/core/day13_fair_comparison/tutorial...,deterministic_synthetic_random_kfold5_tutorial
3,decision_tree,92.5809,116.6940,18.4090,0.2929,5,curriculum/core/day13_fair_comparison/tutorial...,deterministic_synthetic_random_kfold5_tutorial
4,dummy,114.1962,144.0461,23.1084,-0.0674,5,curriculum/core/day13_fair_comparison/tutorial...,deterministic_synthetic_random_kfold5_tutorial


### 3. 生成可追溯、有限定语的报告预览

报告中的数字来自上一步实际表，不提前写死；预览不会声称学习者已完成实验。


In [4]:
best_row = summary.iloc[0]
dummy_row = summary.loc[summary["model"] == "dummy"].iloc[0]
report_text = f"""# Day 14 传统 ML 阶段报告预览（{REPORT_LABEL}）

## 研究问题

在确定性人工回归数据和相同随机 5 折下，五个固定传统基线的折内验证表现怎样？

## 证据来源

- 逐折源文件：`{repo_relative(SOURCE_PATH)}`
- 协议：普通随机 5 折；不是 scaffold split
- 主指标：折内验证 RMSE
- 计时：只保留在 Day 13 运行时内存变量中，不进入预存展示或 CSV fixture
- 行数：{len(fold_metrics)}（模型×折）

## 已执行观察

当前教程中，观察到最低平均 RMSE 的模型是 `{best_row['model']}`，平均 RMSE 为 {best_row['rmse_mean']:.4f}，折间样本标准差为 {best_row['rmse_std']:.4f}。Dummy 的平均 RMSE 为 {dummy_row['rmse_mean']:.4f}。

## 可以得出

{EXECUTION_OBSERVATION}

## 不能得出

这些人工数据数值不代表 ESOL logS 性能，不代表新分子骨架泛化，更不代表真实下游任务强度预测。

## 缺失证据

{MISSING_EVIDENCE_NOTE}
- 未使用 ESOL ECFP；
- 未做 scaffold 或批次分组验证；
- 未评价严格未见测试；
- 未获得真实下游任务数据。

## 下一步

学习张量、MLP 前向、损失和训练循环，再用同一证据协议比较 MLP；不预设 MLP 一定胜出。
"""
report_path = OUTPUT_DIR / "report_preview.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
print("Saved:", repo_relative(report_path))


# Day 14 传统 ML 阶段报告预览（人工教程）

## 研究问题

在确定性人工回归数据和相同随机 5 折下，五个固定传统基线的折内验证表现怎样？

## 证据来源

- 逐折源文件：`curriculum/core/day13_fair_comparison/tutorial_outputs/fold_metrics.csv`
- 协议：普通随机 5 折；不是 scaffold split
- 主指标：折内验证 RMSE
- 计时：只保留在 Day 13 运行时内存变量中，不进入预存展示或 CSV fixture
- 行数：25（模型×折）

## 已执行观察

当前教程中，观察到最低平均 RMSE 的模型是 `ridge`，平均 RMSE 为 18.6088，折间样本标准差为 2.7173。Dummy 的平均 RMSE 为 144.0461。

## 可以得出

课程教程的同折比较、逐折保存、汇总与审计流程已经实际运行。

## 不能得出

这些人工数据数值不代表 ESOL logS 性能，不代表新分子骨架泛化，更不代表真实下游任务强度预测。

## 缺失证据

- 尚未在本人实验目录运行并记录；
- 未使用 ESOL ECFP；
- 未做 scaffold 或批次分组验证；
- 未评价严格未见测试；
- 未获得真实下游任务数据。

## 下一步

学习张量、MLP 前向、损失和训练循环，再用同一证据协议比较 MLP；不预设 MLP 一定胜出。

Saved: curriculum/core/day14_ml_stage_report/tutorial_outputs/report_preview.md


## Checks

确认源文件、汇总、报告预览和边界声明全部存在且一致。


In [5]:
assert len(fold_metrics) == 25
assert set(fold_metrics["model"]) == EXPECTED_MODELS
assert (summary["n_folds"] == 5).all()
assert summary_path.is_file() and report_path.is_file()
assert "人工教程" in report_text
assert "不代表 ESOL" in report_text
assert "真实下游任务" in report_text
assert repo_relative(SOURCE_PATH) in report_text

print("Checks passed: source-audited summary and bounded report preview were generated.")


Checks passed: source-audited summary and bounded report preview were generated.


## Next Steps

课程源只演示如何审计课程 fixture。个人副本会自动审计本人 `learning_outputs/day13_fair_comparison/results/fold_metrics.csv`，但报告仍需亲自逐句审阅；源文件缺失就写“未完成”，绝不能以课程数值、空表或人工填数代替。
